# 02 — Factor Portfolio Returns

This notebook analyzes the five constructed long-short factor portfolios.
The factor returns are already precomputed in the dataset — this notebook focuses on
**understanding their performance characteristics** before using them as targets in the ML models.

**Sections:**
1. Load Data
2. Per-Factor Performance
3. Gross vs Net Returns (Transaction Costs)
4. Benchmark Comparison (vs Fama-French)
5. Cross-Factor Correlation

**Construction Summary:**
- Universe: ~1,600 NYSE common shares per month, bottom 20% by lagged market cap excluded
- Portfolios: top/bottom 30% by signal → value-weighted long-short
- Size neutralization: large-cap and small-cap buckets averaged (all factors except Size)
- Transaction costs: 30 bps × monthly turnover deducted from gross returns

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

from utils import sharpe, max_drawdown

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. Load Data

In [ ]:
df = pd.read_csv('data/FinalMonthlyDataset_ours_ff_macro.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').set_index('date')

FACTOR_LABELS = ['Value', 'Momentum', 'Quality', 'Investment', 'Size']
FACTOR_GROSS  = ['fac_value', 'fac_momentum', 'fac_quality', 'fac_investment', 'fac_size']
FACTOR_NET    = ['fac_value_net', 'fac_momentum_net', 'fac_quality_net', 'fac_investment_net', 'fac_size_net']
FACTOR_TO     = ['to_value', 'to_momentum', 'to_quality', 'to_investment', 'to_size']
COLORS        = ['#2196F3', '#E91E63', '#4CAF50', '#FF9800', '#9C27B0']

print(f'Dataset: {df.shape[0]} months, {df.shape[1]} columns')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')

## 2. Per-Factor Performance

Key metrics for each factor over the full sample. Sharpe ratios are annualized (√12 × monthly).
The per-factor statistics are the baseline against which the timing models in notebook 04 are judged.

In [ ]:
rows = []
for label, gcol, ncol in zip(FACTOR_LABELS, FACTOR_GROSS, FACTOR_NET):
    g = df[gcol].dropna() if gcol in df.columns else pd.Series(dtype=float)
    n = df[ncol].dropna() if ncol in df.columns else pd.Series(dtype=float)
    rows.append({
        'Factor': label,
        'N months': len(n),
        'Gross Sharpe': round(sharpe(g), 3) if len(g) > 1 else np.nan,
        'Net Sharpe':   round(sharpe(n), 3) if len(n) > 1 else np.nan,
        'Gross Ann. Return': round(g.mean() * 12, 4) if len(g) > 1 else np.nan,
        'Net Ann. Return':   round(n.mean() * 12, 4) if len(n) > 1 else np.nan,
        'Ann. Volatility': round(n.std() * np.sqrt(12), 4) if len(n) > 1 else np.nan,
        'Max Drawdown':    round(max_drawdown(n), 4) if len(n) > 1 else np.nan,
    })

perf_df = pd.DataFrame(rows).set_index('Factor')
perf_df

In [ ]:
# Cumulative returns
fig, ax = plt.subplots(figsize=(13, 5))

for ncol, lbl, color in zip(FACTOR_NET, FACTOR_LABELS, COLORS):
    if ncol not in df.columns: continue
    data = df[ncol].dropna()
    cum = (1 + data).cumprod()
    ax.plot(cum.index, cum.values, label=f'{lbl} (net)', color=color, linewidth=1.8)

# EWP benchmark
ewp = df[[c for c in FACTOR_NET if c in df.columns]].mean(axis=1).dropna()
cum_ewp = (1 + ewp).cumprod()
ax.plot(cum_ewp.index, cum_ewp.values, color='black', linewidth=2.2,
        linestyle='--', label='Equal-Weight (EWP)')

ax.axhline(1.0, color='grey', linewidth=0.7, linestyle=':')
ax.set_title('Cumulative Net Factor Returns vs Equal-Weight Portfolio', fontsize=12, fontweight='bold')
ax.set_ylabel('Growth of $1')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Gross vs Net Returns (Transaction Costs)

Net returns deduct **30 bps per unit of monthly portfolio turnover**. High-turnover factors
(e.g., momentum) are penalized more heavily. This matters for the timing model: any strategy
adding factor weight changes will incur additional TC drag.

In [ ]:
tc_rows = []
TC_BPS = 30

for label, gcol, ncol, tocol in zip(FACTOR_LABELS, FACTOR_GROSS, FACTOR_NET, FACTOR_TO):
    if not all(c in df.columns for c in [gcol, ncol, tocol]):
        continue
    gross_ann = df[gcol].mean() * 12
    net_ann   = df[ncol].mean() * 12
    avg_to    = df[tocol].mean()
    tc_rows.append({
        'Factor': label,
        'Avg Monthly Turnover': round(avg_to, 3),
        'Avg Monthly TC (bps)': round(avg_to * TC_BPS, 1),
        'Gross Ann. Return': round(gross_ann, 4),
        'Net Ann. Return':   round(net_ann, 4),
        'TC Drag (ann., bps)': round((gross_ann - net_ann) * 10000, 1),
    })

pd.DataFrame(tc_rows).set_index('Factor')

## 4. Benchmark Comparison (vs Fama-French)

Our constructed factor returns should correlate with the corresponding Fama-French factors
but not be identical — we use a different universe filter and portfolio construction approach.
High correlation validates that we are capturing the same underlying signals.

| Our Factor | FF Benchmark |
|---|---|
| Value | HML (High Minus Low) |
| Momentum | UMD (Up Minus Down) |
| Size | SMB (Small Minus Big) |
| Quality | RMW (Robust Minus Weak) |
| Investment | CMA (Conservative Minus Aggressive) |

In [ ]:
ff_map = {
    'Value':      ('fac_value',      'hml'),
    'Momentum':   ('fac_momentum',   'umd'),
    'Size':       ('fac_size',       'smb'),
    'Quality':    ('fac_quality',    'rmw'),
    'Investment': ('fac_investment', 'cma'),
}

corr_rows = []
for label, (our_col, ff_col) in ff_map.items():
    if not all(c in df.columns for c in [our_col, ff_col]):
        continue
    pair = df[[our_col, ff_col]].dropna()
    corr_rows.append({
        'Factor': label,
        'Our Signal': our_col,
        'FF Benchmark': ff_col,
        'Correlation': round(pair[our_col].corr(pair[ff_col]), 3),
        'N months':    len(pair),
        'Our Sharpe':  round(sharpe(df[our_col].dropna()), 3),
        'FF Sharpe':   round(sharpe(df[ff_col].dropna()), 3),
    })

pd.DataFrame(corr_rows).set_index('Factor')

In [ ]:
# Scatter plots: our factors vs FF benchmarks
valid_pairs = [(lbl, our, ff) for lbl, (our, ff) in ff_map.items()
               if all(c in df.columns for c in [our, ff])]

fig, axes = plt.subplots(1, len(valid_pairs), figsize=(15, 3))

for ax, (lbl, our_col, ff_col), color in zip(axes, valid_pairs, COLORS):
    pair = df[[our_col, ff_col]].dropna()
    corr = pair[our_col].corr(pair[ff_col])
    ax.scatter(pair[ff_col], pair[our_col], alpha=0.35, s=12, color=color)
    ax.set_xlabel(f'FF {ff_col.upper()}', fontsize=9)
    ax.set_ylabel(f'Our {lbl}', fontsize=9)
    ax.set_title(f'{lbl}\n(r = {corr:.2f})', fontsize=9, fontweight='bold')
    ax.grid(alpha=0.3)
    # Add regression line
    m, b = np.polyfit(pair[ff_col], pair[our_col], 1)
    x_line = np.linspace(pair[ff_col].min(), pair[ff_col].max(), 50)
    ax.plot(x_line, m * x_line + b, color='black', linewidth=1.0, linestyle='--')

fig.suptitle('Our Factors vs Fama-French Benchmarks', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Cross-Factor Correlation

Low cross-factor correlation is desirable — it means the five factors provide diversification
to the equal-weight portfolio, and there is room for a timing model to add value by tilting
toward the better-performing factors in any given period.

In [ ]:
net_cols_available = [c for c in FACTOR_NET if c in df.columns]
cross_corr = df[net_cols_available].corr()
cross_corr.index   = [lbl for lbl, c in zip(FACTOR_LABELS, FACTOR_NET) if c in df.columns]
cross_corr.columns = cross_corr.index

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cross_corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)

n = len(cross_corr)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(cross_corr.columns, rotation=30, ha='right')
ax.set_yticklabels(cross_corr.index)

for i in range(n):
    for j in range(n):
        val = cross_corr.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=10, color='black' if abs(val) < 0.6 else 'white', fontweight='bold')

ax.set_title('Cross-Factor Correlation (Net Returns)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey observation: low to moderate cross-factor correlations support diversification.')
print('Average off-diagonal correlation:', round(cross_corr.values[cross_corr.values < 0.999].mean(), 3))